In [2]:
import pandas as pd
import numpy as np
import re
import string
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
# Load cleaned training data from Week 6
train_df = pd.read_parquet('cleaned_train.parquet')
print(f"Training data shape: {train_df.shape}")
print(f"Columns: {train_df.columns.tolist()}")
print(f"Number of unique users: {train_df['user_id'].nunique()}")
print(f"Number of unique items: {train_df['item_id'].nunique()}")

Training data shape: (26580, 3)
Columns: ['item_id', 'user_id', 'rating']
Number of unique users: 1389
Number of unique items: 932


In [4]:
# Load metadata
meta_df = pd.read_parquet('data/meta_video_games.parquet')
print(f"Original metadata shape: {meta_df.shape}")
print(f"Metadata columns: {meta_df.columns.tolist()}")

# Get items that appear in training data
items_in_train = set(train_df['item_id'].unique())
print(f"\nItems in training data: {len(items_in_train)}")

# Filter metadata to only items in training
meta_df = meta_df[meta_df['item_id'].isin(items_in_train)].copy()
print(f"Filtered metadata shape: {meta_df.shape}")

Original metadata shape: (137269, 8)
Metadata columns: ['item_id', 'title', 'description', 'features', 'categories', 'details', 'rating_number', 'average_rating']

Items in training data: 932
Filtered metadata shape: (932, 8)


In [5]:
# Check what's in the description column
print("First 3 descriptions:")
for i in range(min(3, len(meta_df))):
    desc = meta_df['description'].iloc[i]
    print(f"\n{i+1}. Type: {type(desc)}")
    print(f"   Content: {desc}")

First 3 descriptions:

1. Type: <class 'numpy.ndarray'>
   Content: ['Amazon.com' 'Long recognized as role-playing games par excellence, the'
 'Final Fantasy'
 'series gets a technological makeover in this installment (and series debut on the PlayStation). Shedding the two-dimensional graphics and limited sound capabilities of its predecessors,'
 'Final Fantasy VII'
 'features lush 3-D graphics, beautifully animated "movie" sequences, and soundtrack-quality music. Coupled with the game\'s intricate storyline, endearing characters, and immense yet highly imaginative world, these new advancements make for a quite an engrossing experience.'
 'The story of' 'Final Fantasy VII'
 'centers around a solider named Cloud Strife, who joins forces with Avalanche, a group of resistance fighters, to take down an evil mega-corporation known as Shinra. (The fate of the world hangs in the balance, of course.) Truly epic in scope, this four-disc game requires a considerable amount of time to complete---

In [6]:
def convert_to_string(desc):
    """Convert any description to a string safely"""
    try:
        # Check if it's NaN
        if pd.isna(desc):
            return ''
        
        # If it's a string already
        if isinstance(desc, str):
            return desc
        
        # If it's a list or array, join all elements
        if hasattr(desc, '__iter__') and not isinstance(desc, str):
            # Convert each element to string and join
            return ' '.join(str(x) for x in desc)
        
        # Fallback: convert to string
        return str(desc)
    except Exception as e:
        # If anything fails, return empty string
        print(f"Error converting: {desc[:50] if hasattr(desc, '__len__') else desc} - {e}")
        return ''

# Apply conversion
print("Converting descriptions to strings...")
meta_df['description_text'] = meta_df['description'].apply(convert_to_string)

# Check results
print(f"\nConverted {len(meta_df)} descriptions")
print(f"Sample: {meta_df['description_text'].iloc[0][:200]}...")

Converting descriptions to strings...
Error converting: ['Amazon.com' 'Long recognized as role-playing games par excellence, the'
 'Final Fantasy'
 'series gets a technological makeover in this installment (and series debut on the PlayStation). Shedding the two-dimensional graphics and limited sound capabilities of its predecessors,'
 'Final Fantasy VII'
 'features lush 3-D graphics, beautifully animated "movie" sequences, and soundtrack-quality music. Coupled with the game\'s intricate storyline, endearing characters, and immense yet highly imaginative world, these new advancements make for a quite an engrossing experience.'
 'The story of' 'Final Fantasy VII'
 'centers around a solider named Cloud Strife, who joins forces with Avalanche, a group of resistance fighters, to take down an evil mega-corporation known as Shinra. (The fate of the world hangs in the balance, of course.) Truly epic in scope, this four-disc game requires a considerable amount of time to complete---this reviewe

In [7]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Use scikit-learn's built-in stopwords
STOPWORDS = set(ENGLISH_STOP_WORDS)

# Add video game specific words (optional)
game_words = {'game', 'games', 'video', 'play', 'playing', 'player', 'players'}
STOPWORDS.update(game_words)

print(f"Total stopwords: {len(STOPWORDS)}")
print(f"Sample stopwords: {list(STOPWORDS)[:20]}")

Total stopwords: 325
Sample stopwords: ['and', 'toward', 'may', 'itself', 'one', 'here', 'can', 'thick', 'due', 'if', 'perhaps', 'please', 'inc', 'whereupon', 'fifty', 'yourselves', 'besides', 'mine', 'thin', 'top']


In [8]:
def preprocess_text(text):
    """
    Preprocess text by:
    1. Converting to lowercase
    2. Removing punctuation
    3. Removing numbers
    4. Removing stopwords
    5. Keeping words with length > 2
    """
    if not isinstance(text, str) or text == '':
        return ''
    
    # Lowercase
    text = text.lower()
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Split into words
    words = text.split()
    
    # Remove stopwords and short words
    words = [w for w in words if w not in STOPWORDS and len(w) > 2]
    
    # Join back
    return ' '.join(words)

# Test the function
test_text = meta_df['description_text'].iloc[0]
print(f"Original: {test_text[:150]}...")
print(f"\nProcessed: {preprocess_text(test_text)[:150]}...")

Original: ...

Processed: ...


In [9]:
# Apply preprocessing
print("Applying preprocessing to all descriptions...")
meta_df['processed_text'] = meta_df['description_text'].apply(preprocess_text)

# Calculate vocabulary sizes
all_text_before = ' '.join(meta_df['description_text'])
all_text_after = ' '.join(meta_df['processed_text'])

vocab_before = len(set(all_text_before.split()))
vocab_after = len(set(all_text_after.split()))

print(f"\nVocabulary size BEFORE preprocessing: {vocab_before:,}")
print(f"Vocabulary size AFTER preprocessing: {vocab_after:,}")
print(f"Reduction: {((vocab_before - vocab_after) / vocab_before * 100):.1f}%")

# Check how many items have content
non_empty_count = (meta_df['processed_text'] != '').sum()
print(f"\nItems with non-empty text after preprocessing: {non_empty_count} out of {len(meta_df)} ({non_empty_count/len(meta_df)*100:.1f}%)")

# Show examples
print("\nExamples of processed text:")
sample = meta_df[meta_df['processed_text'] != ''].head(3)
for i, row in sample.iterrows():
    print(f"\nItem: {row['item_id']}")
    print(f"Processed: {row['processed_text'][:100]}...")

Applying preprocessing to all descriptions...

Vocabulary size BEFORE preprocessing: 10,212
Vocabulary size AFTER preprocessing: 6,386
Reduction: 37.5%

Items with non-empty text after preprocessing: 391 out of 932 (42.0%)

Examples of processed text:

Item: B00004Y57G
Processed: final fantasy ixgreatest hits...

Item: B00006IJJK
Processed: samus enters mysterious derelict ship unexplored world tallon investigate space pirate activities th...

Item: B00008URUA
Processed: taking place years yuna...


In [10]:
# Keep only items that have non-empty processed text
items_with_text = meta_df[meta_df['processed_text'] != ''].copy()
print(f"Items with text: {len(items_with_text)} out of {len(meta_df)} ({len(items_with_text)/len(meta_df)*100:.1f}%)")

if len(items_with_text) == 0:
    print("\nWARNING: No items have text after preprocessing!")
    print("Using raw text instead...")
    items_with_text = meta_df.copy()
    items_with_text['processed_text'] = items_with_text['description_text']

Items with text: 391 out of 932 (42.0%)


In [12]:
# Create TF-IDF vectors
print("Creating TF-IDF vectors...")
tfidf_vectorizer = TfidfVectorizer(max_features=500, stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(items_with_text['processed_text'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Number of features (unique terms): {len(tfidf_vectorizer.get_feature_names_out())}")

# Show top terms
feature_names = tfidf_vectorizer.get_feature_names_out()
print("\nTop 20 most important terms (by average TF-IDF score):")
avg_scores = tfidf_matrix.mean(axis=0).tolist()[0]
top_indices = np.argsort(avg_scores)[-20:][::-1]
for idx in top_indices:
    print(f"  {feature_names[idx]}: {avg_scores[idx]:.4f}")

Creating TF-IDF vectors...
TF-IDF matrix shape: (391, 500)
Number of features (unique terms): 500

Top 20 most important terms (by average TF-IDF score):
  new: 0.0536
  world: 0.0405
  experience: 0.0276
  xbox: 0.0258
  nintendo: 0.0247
  playstation: 0.0246
  adventure: 0.0234
  wii: 0.0218
  series: 0.0217
  war: 0.0216
  action: 0.0214
  set: 0.0212
  city: 0.0208
  time: 0.0208
  controller: 0.0200
  story: 0.0196
  gameplay: 0.0193
  content: 0.0187
  enemies: 0.0185
  power: 0.0175


In [13]:
# Create dictionary mapping item_id to its TF-IDF vector
item_to_tfidf = {}
for i, item_id in enumerate(items_with_text['item_id']):
    item_to_tfidf[item_id] = tfidf_matrix[i]

print(f"Created mapping for {len(item_to_tfidf)} items")

Created mapping for 391 items


In [14]:
from sklearn.metrics.pairwise import cosine_similarity

def get_similar_items(item_id, n=5):
    if item_id not in item_to_tfidf:
        return []
    
    target = item_to_tfidf[item_id]
    similarities = []
    
    for other_id, other_vec in item_to_tfidf.items():
        if other_id != item_id:
            if hasattr(target, 'toarray'):
                target_dense = target.toarray()[0]
                other_dense = other_vec.toarray()[0]
                sim = cosine_similarity([target_dense], [other_dense])[0][0]
            else:
                sim = cosine_similarity([target], [other_vec])[0][0]
            similarities.append((other_id, sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:n]

# Pick a sample item (one with meaningful text)
sample_items_with_text = items_with_text[items_with_text['processed_text'].str.len() > 50]['item_id'].tolist()

if sample_items_with_text:
    sample_item = sample_items_with_text[0]
    sample_desc = items_with_text[items_with_text['item_id'] == sample_item]['description_text'].iloc[0]
    
    print(f"Sample Item: {sample_item}")
    print(f"Description: {sample_desc[:200]}...")
    
    # Find similar items
    similar_items = get_similar_items(sample_item, n=5)
    
    print("\nTop 5 Similar Items:")
    for i, (item, sim) in enumerate(similar_items):
        similar_desc = items_with_text[items_with_text['item_id'] == item]['description_text'].iloc[0]
        print(f"\n{i+1}. Item: {item}")
        print(f"   Similarity: {sim:.4f}")
        print(f"   Description: {similar_desc[:100]}...")
else:
    similar_items = []
    sample_item = "No item found"
    print("No suitable sample item found")

Sample Item: B00006IJJK
Description: Samus enters a mysterious derelict ship on the unexplored world of Tallon IV to investigate Space Pirate activities. She has thwarted their dastardly efforts before. She stopped them from amassing an ...

Top 5 Similar Items:

1. Item: B000FQBPDU
   Similarity: 0.2512
   Description: You ARE Samus with Wii control! By moving around with the Nunchuk and aiming Samus's gun with the Wi...

2. Item: B075XLGQ4V
   Similarity: 0.2498
   Description: Brave the hostile terrain of an alien planet teeming with vicious life forms as legendary bounty hun...

3. Item: B001EYUXIK
   Similarity: 0.2229
   Description: Star Wars Battlefront II adds all-new space combat, playable Jedi characters, and never-before-seen ...

4. Item: B002JTX5UM
   Similarity: 0.2167
   Description: In deep space, on the frontier of human colonization, an ancient pyramid is discovered. When archaeo...

5. Item: B07WS18ZS3
   Similarity: 0.2082
   Description: For PlayStation owners, Ba

In [15]:
# Create a summary of Week 9 results
week9_results = {
    'Metric': [
        'Total items in dataset',
        'Items with descriptions',
        'Items with text after preprocessing',
        'Vocabulary size (before)',
        'Vocabulary size (after)',
        'Vocabulary reduction (%)',
        'TF-IDF features',
        'Sample item for similarity',
        'Top similar item',
        'Top similarity score'
    ],
    'Value': [
        len(meta_df),
        len(meta_df[meta_df['description_text'].str.len() > 0]),
        len(items_with_text),
        vocab_before,
        vocab_after,
        f"{((vocab_before - vocab_after) / vocab_before * 100):.1f}%",
        tfidf_matrix.shape[1],
        sample_item if 'sample_item' in dir() else 'N/A',
        similar_items[0][0] if 'similar_items' in dir() and similar_items else 'N/A',
        f"{similar_items[0][1]:.4f}" if 'similar_items' in dir() and similar_items else 'N/A'
    ]
}

week9_df = pd.DataFrame(week9_results)
week9_df.to_csv('week9_results.csv', index=False)
print("Week 9 results saved to week9_results.csv")
print("\n" + week9_df.to_string(index=False))

Week 9 results saved to week9_results.csv

                             Metric      Value
             Total items in dataset        932
            Items with descriptions        391
Items with text after preprocessing        391
           Vocabulary size (before)      10212
            Vocabulary size (after)       6386
           Vocabulary reduction (%)      37.5%
                    TF-IDF features        500
         Sample item for similarity B00006IJJK
                   Top similar item B000FQBPDU
               Top similarity score     0.2512


In [16]:
import pickle

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

with open('item_to_tfidf.pkl', 'wb') as f:
    pickle.dump(item_to_tfidf, f)

with open('items_with_text.pkl', 'wb') as f:
    pickle.dump(items_with_text, f)

print("Saved files:")
print("  - tfidf_vectorizer.pkl")
print("  - item_to_tfidf.pkl")
print("  - items_with_text.pkl")
print("  - week9_results.csv")

Saved files:
  - tfidf_vectorizer.pkl
  - item_to_tfidf.pkl
  - items_with_text.pkl
  - week9_results.csv
